# Spaceship Titanic — entrenar y exportar

Este notebook entrena el clasificador del reto y **exporta el artefacto de despliegue**.

Es el hermano de `01-entrenar-y-exportar.ipynb`, que hace lo mismo para el modelo de precios
de vivienda. Mismo contrato, mismo servicio, mismo tablero: lo único que cambia es el
problema.

```
artifacts/
├── pipeline.joblib     el Pipeline COMPLETO: columnas derivadas +
│                       preprocesamiento + clasificador
├── derivadas.py        el módulo con las columnas derivadas  (*)
├── metadata.json       el CONTRATO, en texto legible
└── example.json        un input válido, su clase y sus probabilidades
```

(*) Este archivo es nuevo respecto al modelo de casas, y la sección 5 explica por qué tiene
que existir. Resumen: `joblib` no guarda el código de una función, guarda su nombre.

---

## Antes de empezar: sube los datos

Arrastra la carpeta `data` con el `train.csv` **de Spaceship Titanic** (el de Kaggle, no el
de casas). Debe quedarte así:

```
/content/
└── data/
    └── train.csv
```

## 0. El entorno tiene que coincidir con el del servidor

Colab trae sus propias versiones, y **no son las que tu servidor tiene instaladas**.

`joblib` no es un formato estable: un modelo exportado con una versión y cargado con otra
puede fallar al abrirse o —peor— abrirse y devolver números distintos sin avisar.

Aquí se añade **xgboost** a los pines del curso. El artefacto lleva un `XGBClassifier`
dentro, así que la instancia necesita la librería para deserializarlo: sin ella, el servicio
truena al arrancar con `No module named 'xgboost'`.

In [ ]:
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules

# Las mismas versiones que backend/requirements.txt. Si cambias una, cambia las dos.
PINES = {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scikit-learn": "1.5.2",
    "joblib": "1.4.2",
    "xgboost": "2.1.3",
}

print(f"Python de este entorno: {sys.version.split()[0]}")

if EN_COLAB:
    print("Instalando las versiones del proyecto (tarda un minuto)...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"]
        + [f"{p}=={v}" for p, v in PINES.items()],
        capture_output=True,
        text=True,
    )
    if r.returncode == 0:
        print("Listo.")
    else:
        # Sin check=True: un traceback de subprocess no dice nada, y el error
        # de pip dice exactamente que paquete no se pudo instalar.
        print()
        print("La instalacion FALLO. Esto es lo que dijo pip:")
        print()
        print((r.stderr or r.stdout)[-1500:])
else:
    print("No estas en Colab: se usan las versiones del entorno virtual del proyecto.")

### Comprueba que quedaron las versiones correctas

Si esta celda te pide reiniciar, hazlo: **Entorno de ejecución → Reiniciar sesión**, y vuelve
a correr desde aquí.

In [ ]:
import importlib.metadata as md

problemas = []
for paquete, esperada in PINES.items():
    try:
        instalada = md.version(paquete)
    except md.PackageNotFoundError:
        problemas.append(f"{paquete}: no instalado")
        continue
    marca = "OK  " if instalada == esperada else "MAL "
    print(f"  {marca} {paquete:<14} {instalada}   (se espera {esperada})")
    if instalada != esperada:
        problemas.append(f"{paquete}: {instalada} en lugar de {esperada}")

print()
if problemas:
    print("Reinicia el entorno de ejecucion y vuelve a correr desde la celda anterior:")
    print("   Entorno de ejecucion -> Reiniciar sesion")
else:
    print("El entorno coincide con el del servidor.")

## 1. Lo que necesitamos

In [ ]:
import hashlib
import json
import pathlib
import sys
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler
from xgboost import XGBClassifier

## 2. Dónde están los datos

Esta función busca `data/train.csv` hacia arriba desde donde estés. Funciona igual en Colab,
en la raíz del proyecto o en la carpeta `notebooks/`.

Si truena, es que no subiste la carpeta `data`.

In [ ]:
def encontrar_raiz():
    """Localiza la carpeta del proyecto buscando data/train.csv hacia arriba.

    No usa __file__ porque en Colab no existe.
    """
    try:
        candidatos = [pathlib.Path(__file__).resolve().parent.parent]
    except NameError:
        candidatos = []
    aqui = pathlib.Path.cwd()
    candidatos += [aqui, *aqui.parents]

    for base in candidatos:
        if (base / "data" / "train.csv").exists():
            return base

    raise FileNotFoundError(
        "No encuentro data/train.csv.\n\n"
        "    Si estas en Colab: arrastra la carpeta 'data' completa al panel\n"
        "    de archivos de la izquierda y vuelve a correr esta celda.\n"
    )


RAIZ = encontrar_raiz()
DATOS = RAIZ / "data" / "train.csv"
ARTEFACTOS = RAIZ / "artifacts"
ARTEFACTOS.mkdir(exist_ok=True)

df_crudo = pd.read_csv(DATOS)

if "Transported" not in df_crudo.columns:
    raise RuntimeError(
        f"{DATOS} no tiene la columna 'Transported'.\n"
        "    Parece el CSV de otro dataset. Este notebook es para Spaceship Titanic."
    )

# Tiene que ser el train.csv CRUDO de Kaggle, no el train_processed.csv del
# avance 1.
#
# El procesado ya viene codificado --HomePlanet_Europa, Cabin_Deck_B...-- y esas
# columnas no sirven aqui por una razon de fondo: son el resultado del
# preprocesamiento, y el preprocesamiento es justo lo que tiene que quedar
# DENTRO del artefacto. Un servicio que pidiera 'HomePlanet_Europa' le estaria
# pasando al usuario el trabajo del modelo.
CRUDAS = ["PassengerId", "HomePlanet", "CryoSleep", "Cabin", "Destination", "Age"]
faltantes = [c for c in CRUDAS if c not in df_crudo.columns]
if faltantes:
    raise RuntimeError(
        f"A {DATOS} le faltan columnas del CSV original: {', '.join(faltantes)}.\n\n"
        "    Esto pasa cuando se usa train_processed.csv en lugar del train.csv\n"
        "    de Kaggle. Este notebook necesita el CRUDO: el preprocesamiento lo\n"
        "    hace el pipeline, no tu.\n\n"
        f"    Lo que si trae el archivo: {', '.join(df_crudo.columns[:8])}..."
    )

print(f"proyecto : {RAIZ}")
print(f"datos    : {DATOS}  ({len(df_crudo)} filas)")

## 3. Las trece features — una decisión de producto

No son las que dan el mejor recall. Son **las que alguien puede contestar sobre un pasajero**
sin tener el dataset completo delante.

Tres cosas que se quedan fuera, y por qué:

| Columna | Por qué no está |
|---|---|
| `Name`, `PassengerId` | Identificadores únicos, sin valor predictivo |
| `VIP` | 97.7% en `False`; el crosstab no mostró diferencia concluyente |
| `Cabin` | Se abre en `Cabin_Deck` / `Cabin_Num` / `Cabin_Side`, que sí se pueden agrupar |

Y una que **sí** está aunque no venga en el CSV: `GroupSize`. Sale del prefijo de
`PassengerId`, y es información que el pasajero conoce —con cuánta gente viaja—.

> **Lo que esto cuesta.** En el avance 1, `HomePlanet` se imputaba con la moda **del grupo de
> viaje**. Eso aquí no se puede: el servicio predice de un pasajero a la vez y no tiene el
> grupo delante. Se imputa con la moda global. Es una pérdida real de señal, y es el precio
> de que el modelo funcione fuera del notebook. Vale la pena decirlo en voz alta en lugar de
> que aparezca como una diferencia inexplicable entre tus métricas y las del servicio.

In [ ]:
MODEL_VERSION = "1.0.0"
SEMILLA = 42
TARGET = "Transported"

# Numericas que el usuario contesta tal cual.
NUMERICAS = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "Cabin_Num",
    "GroupSize",
]
CATEGORICAS = ["HomePlanet", "Destination", "Cabin_Deck", "Cabin_Side"]
BOOLEANAS = ["CryoSleep"]

FEATURES = NUMERICAS + CATEGORICAS + BOOLEANAS

# Como se llama cada clase para una persona. El modelo predice 0 y 1; nadie
# quiere leer eso en una pantalla.
ETIQUETAS = {"0": "No transportado", "1": "Transportado"}
CLASE_POSITIVA = 1

print(f"{len(FEATURES)} features: {', '.join(FEATURES)}")

## 4. Del CSV al vocabulario del contrato

`Cabin` viene como `B/0/P` y `PassengerId` como `0001_01`. Abrir esas dos columnas **no es
preprocesamiento del modelo**: es leer el archivo. Por eso vive aquí y no dentro del pipeline.

La frontera es esta: lo que hace falta para **construir las features del contrato** se hace
aquí; lo que hace falta para **convertir esas features en números** viaja dentro del
artefacto.

In [ ]:
def preparar(df):
    """Del CSV de Kaggle a las columnas que declara el contrato."""
    df = df.copy()

    partes = df["Cabin"].str.split("/", expand=True)
    df["Cabin_Deck"] = partes[0]
    df["Cabin_Num"] = pd.to_numeric(partes[1], errors="coerce")
    df["Cabin_Side"] = partes[2]

    # El PassengerId es '<grupo>_<numero dentro del grupo>'. Cuantos comparten
    # grupo es una señal util --la gente viaja junta-- y el pasajero la sabe.
    grupo = df["PassengerId"].astype(str).str.split("_").str[0]
    df["GroupSize"] = grupo.map(grupo.value_counts()).astype(float)

    return df


df = preparar(df_crudo)
print(df[FEATURES].dtypes.to_string())
print()
print("faltantes por feature:")
print(df[FEATURES].isna().sum()[lambda s: s > 0].to_string() or "  ninguno")

## 5. `derivadas.py` — el módulo que viaja con el artefacto

Esta es la parte nueva respecto al modelo de casas, y vale la pena entenderla bien.

El pipeline necesita columnas que el usuario no escribe: el gasto total, si gastó algo, si
es menor de edad, si viaja solo, y el `log1p` de las cuentas de consumo. Todo eso es
**conocimiento del modelo**, así que tiene que viajar dentro del artefacto.

La forma natural de meterlo en un `Pipeline` es un `FunctionTransformer`. Y ahí aparece el
problema:

```
joblib.dump(pipeline)   ──▶  guarda la REFERENCIA 'derivadas.derivar',
                             NO el código de la función

joblib.load(pipeline)   ──▶  intenta  import derivadas
                             ¿no existe?  ModuleNotFoundError
```

Si definieras `derivar()` en una celda de este notebook, la referencia sería
`__main__.derivar` y el servidor fallaría con
`Can't get attribute 'derivar' on module '__main__'`.

**La solución:** el módulo se escribe como archivo dentro de `artifacts/`, y el servicio pone
esa carpeta en `sys.path` antes de deserializar. El artefacto sigue siendo autocontenido —
solo que ahora es una carpeta, no un archivo suelto.

Una regla que sale de esto y que conviene recordar: **toda la derivación es por fila.** Ni un
solo `groupby`. Tiene que dar el mismo resultado con 8 693 filas al entrenar que con una sola
al servir.

In [ ]:
CODIGO_DERIVADAS = r'''"""Columnas derivadas del modelo de Spaceship Titanic.

ESTE ARCHIVO VIAJA CON EL ARTEFACTO, y no es un capricho.

joblib no serializa el CODIGO de una funcion: guarda una referencia con su
nombre completo ('derivadas.derivar'). Si el pipeline lleva dentro un
FunctionTransformer y este modulo no se puede importar al cargarlo, joblib
truena con "No module named 'derivadas'".

Por eso el notebook lo escribe dentro de artifacts/ junto a pipeline.joblib, y
el servicio pone esa carpeta en sys.path antes de deserializar. El artefacto
sigue siendo autocontenido: es una carpeta, no un archivo suelto.

Y por eso NADA de aqui vive en backend/. Toda la transformacion que el modelo
necesita viaja con el modelo. El servicio le pasa un DataFrame crudo --las
features del contrato, tal como las escribio el usuario-- y no sabe que estas
columnas existen.
"""

import numpy as np
import pandas as pd

# Las cinco cuentas de consumo a bordo.
GASTOS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# Las que se comprimen con log1p: tienen sesgo de entre 6 y 12 en crudo.
# log1p mapea el 0 a 0 --y aqui hay muchisimos ceros, por el criosueño--
# conserva el orden y comprime la cola sin borrar los extremos.
A_COMPRIMIR = GASTOS + ["TotalSpent"]

# Cubiertas de la nave. La T tiene un puñado de registros en el dataset real:
# dejarla sola invita al modelo a aprender una relacion espuria sobre cinco
# filas, asi que se funde con la D, su vecina.
DECK_RARA, DECK_DESTINO = "T", "D"


def _a_binaria(serie):
    """Lleva a 1.0 / 0.0 / NaN una columna que puede llegar de varias formas.

    Al entrenar, pandas lee la columna del CSV como objeto con True, False y
    NaN mezclados. Al servir, el backend ya la normalizo a bool de Python. Y
    un cliente escrito en otro lenguaje puede mandar "true" o 1.

    Las cuatro formas quieren decir lo mismo. Normalizar aqui --y no en el
    servicio-- es lo que hace que el mismo pipeline sirva para las dos rutas.
    """
    mapa = {
        True: 1.0, False: 0.0,
        "True": 1.0, "False": 0.0,
        "true": 1.0, "false": 0.0,
        "1": 1.0, "0": 0.0,
    }
    # True, 1 y 1.0 tienen el mismo hash en Python, asi que las tres formas
    # numericas caen en la misma entrada del diccionario sin listarlas todas.
    return pd.Series(
        [mapa.get(v, np.nan) for v in serie], index=serie.index, dtype=float
    )


def derivar(X):
    """Agrega las columnas que el modelo necesita y que el usuario no escribe.

    Recibe las features del contrato en crudo; devuelve un DataFrame con esas
    mismas columnas mas las derivadas. No imputa ni escala: eso es trabajo del
    ColumnTransformer que viene despues. Aqui solo se construyen columnas.

    Tiene que funcionar igual con 8693 filas al entrenar y con UNA sola al
    servir. Por eso no hay ni un groupby: toda la derivacion es por fila.
    """
    X = pd.DataFrame(X).copy()

    # --- CryoSleep -------------------------------------------------------
    # Faltante informativo: un pasajero en criosueño no puede gastar. Si
    # CryoSleep falta pero hay gasto registrado, CryoSleep era False.
    X["CryoSleep"] = _a_binaria(X["CryoSleep"])
    gasto_conocido = X[GASTOS].sum(axis=1, skipna=True) > 0
    X.loc[X["CryoSleep"].isna() & gasto_conocido, "CryoSleep"] = 0.0

    # Y al reves: si esta en criosueño, un gasto faltante es un cero, no un
    # desconocido. Imputar la mediana ahi seria inventar consumo.
    en_cryo = X["CryoSleep"] == 1.0
    for c in GASTOS:
        X.loc[en_cryo & X[c].isna(), c] = 0.0

    # --- Gasto total y si gasto algo -------------------------------------
    X["TotalSpent"] = X[GASTOS].sum(axis=1, skipna=True)
    X["HasSpent"] = (X["TotalSpent"] > 0).astype(float)

    # --- Señales de contexto ---------------------------------------------
    X["IsChild"] = (pd.to_numeric(X["Age"], errors="coerce") < 15).astype(float)
    X["IsAlone"] = (pd.to_numeric(X["GroupSize"], errors="coerce") == 1).astype(float)

    # --- Cubierta rara ----------------------------------------------------
    X["Cabin_Deck"] = X["Cabin_Deck"].replace(DECK_RARA, DECK_DESTINO)

    # --- Compresion del sesgo --------------------------------------------
    for c in A_COMPRIMIR:
        X[f"{c}_log"] = np.log1p(pd.to_numeric(X[c], errors="coerce").clip(lower=0))

    return X.drop(columns=A_COMPRIMIR)


# Las columnas que salen de derivar(), agrupadas por como hay que tratarlas.
# El ColumnTransformer del pipeline se construye con estas listas.
CONTINUAS = [f"{c}_log" for c in A_COMPRIMIR] + ["Age", "Cabin_Num", "GroupSize"]
NOMINALES = ["HomePlanet", "Destination", "Cabin_Deck", "Cabin_Side"]
BINARIAS = ["CryoSleep", "HasSpent", "IsChild", "IsAlone"]

# De que feature del contrato salio cada columna derivada.
#
# Sirve para una sola cosa, y es importante: devolver las importancias del
# modelo al vocabulario del usuario. El modelo ve 'TotalSpent_log' y
# 'HomePlanet_Europa'; la Model Card tiene que hablar de 'RoomService' y
# 'HomePlanet', que son los campos que la persona llena.
#
# Una columna que resume varias features se reparte entre todas ellas.
# Las que no aparecen aqui se atribuyen a la feature de su mismo nombre.
ORIGEN = {
    **{f"{c}_log": [c] for c in GASTOS},
    "TotalSpent_log": GASTOS,
    "HasSpent": GASTOS,
    "IsChild": ["Age"],
    "IsAlone": ["GroupSize"],
}
'''

ruta_modulo = ARTEFACTOS / "derivadas.py"
ruta_modulo.write_text(CODIGO_DERIVADAS)

# Se importa DESDE el archivo recien escrito, no desde esta celda.
#
# Es justo el punto: si importaramos una funcion definida aqui, joblib la
# guardaria como '__main__.derivar' y el servidor no podria encontrarla.
if str(ARTEFACTOS) not in sys.path:
    sys.path.insert(0, str(ARTEFACTOS))

import importlib

import derivadas

importlib.reload(derivadas)  # por si ya estaba cargado de una corrida anterior

print(f"escrito: {ruta_modulo}")
print(f"referencia que va a guardar joblib: {derivadas.derivar.__module__}.derivar")
assert derivadas.derivar.__module__ == "derivadas", (
    "la funcion quedo en __main__: el artefacto no cargaria en el servidor"
)

# Prueba rapida: la derivacion tiene que dar lo mismo con muchas filas y con una.
muchas = derivadas.derivar(df[FEATURES])
una = derivadas.derivar(df[FEATURES].iloc[[0]])
assert list(muchas.columns) == list(una.columns), "la derivacion depende del numero de filas"
print(f"\ncolumnas despues de derivar ({len(muchas.columns)}):")
print("  " + ", ".join(muchas.columns))

## 6. El pipeline completo — la pieza central

Mira dónde vive cada cosa:

```
       entrada cruda del contrato (13 columnas)
                     │
   ┌─────────────────▼─────────────────┐
   │ derivar        TotalSpent, HasSpent, IsChild, IsAlone,
   │                log1p de los consumos, arreglo de CryoSleep
   ├───────────────────────────────────┤
   │ preproceso     imputar + RobustScaler   (continuas)
   │                imputar + OneHot         (nominales)
   │                imputar                  (binarias)
   ├───────────────────────────────────┤
   │ estimador      XGBClassifier
   └───────────────────────────────────┘
                     │
                  0 ó 1
```

Todo en **un solo objeto**. El servicio le pasa un `DataFrame` crudo y recibe una clase, sin
saber nada de imputaciones, escalados ni logaritmos.

**`RobustScaler` y no `StandardScaler`**: después del `log1p` siguen quedando valores extremos,
sobre todo en `Age`. `RobustScaler` usa mediana y rango intercuartil, así que los extremos
pesan menos.

**`handle_unknown="ignore"`** en el one-hot: si algún día llega una cubierta que el modelo no
vio, la fila se codifica en ceros y la predicción sale, en lugar de tumbar el servicio. El
aviso de "valor no visto" ya lo da la validación del contrato.

> **Un detalle del avance 1 que aquí queda arreglado.** En aquel notebook el `RobustScaler` se
> ajustaba sobre una copia (`X[continuas] = scaler.fit_transform(...)`), pero el `df_train` que
> después alimentó a los modelos nunca lo recibió: **entrenaron con los datos sin escalar**. No
> cambió tus métricas —los árboles son invariantes a un escalado monótono por columna, y por eso
> nadie lo notó— pero sí habría cambiado todo con un modelo lineal o un KNN. Metido dentro del
> `Pipeline` el problema no puede volver a ocurrir: `fit` y `transform` van juntos o no van.

In [ ]:
def construir_preproceso():
    """Cada grupo de columnas con el tratamiento que le toca.

    Las listas salen de derivadas.py --el modulo que las CREA es el que sabe
    como se llaman-- y no estan escritas otra vez aqui. Si mañana agregas una
    columna derivada, la declaras en un solo sitio.
    """
    return ColumnTransformer(
        transformers=[
            (
                "cont",
                Pipeline(
                    [
                        ("imputar", SimpleImputer(strategy="median")),
                        ("escalar", RobustScaler()),
                    ]
                ),
                derivadas.CONTINUAS,
            ),
            (
                "nom",
                Pipeline(
                    [
                        ("imputar", SimpleImputer(strategy="most_frequent")),
                        (
                            "codificar",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                derivadas.NOMINALES,
            ),
            # Las binarias ya estan en 0/1: solo falta rellenar los huecos.
            # Escalarlas no aportaria nada y haria el artefacto mas dificil de leer.
            ("bin", SimpleImputer(strategy="most_frequent"), derivadas.BINARIAS),
        ]
    )


# Los hiperparametros que gano el GridSearchCV del avance 3:
# 240 combinaciones x 5 folds estratificados, optimizando recall.
# Mejor recall en CV: 0.9038
PARAMS_XGB = dict(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.08,
    min_child_weight=1,
    subsample=0.8,
    # scale_pos_weight NO corrige desbalance: las clases estan a 50.4/49.6.
    # Desplaza a proposito el punto de operacion hacia el recall, porque los
    # dos errores no cuestan lo mismo. Es una decision de producto metida en
    # la funcion de perdida, y por eso viaja dentro del artefacto.
    scale_pos_weight=2,
    random_state=SEMILLA,
    eval_metric="logloss",
)


def armar(estimador):
    """El pipeline completo alrededor del estimador que le pases.

    Que sea un parametro y no algo fijo es lo que permite comparar modelos mas
    abajo SIN cambiar el preprocesamiento: si cada candidato se preparara los
    datos a su manera, la comparacion no diria nada.
    """
    return Pipeline(
        [
            ("derivar", FunctionTransformer(derivadas.derivar, validate=False)),
            ("preproceso", construir_preproceso()),
            ("estimador", estimador),
        ]
    )


def construir_pipeline():
    return armar(XGBClassifier(**PARAMS_XGB))

## 7. Tres conjuntos, no dos

Entrenamiento para aprender, validación para decidir, prueba para reportar. **La prueba se
toca una sola vez**, al final.

`stratify=y` mantiene la proporción de clases en los tres. Sin eso, con un split desafortunado
la validación puede quedar con un balance distinto al del entrenamiento y las métricas dejan
de ser comparables.

In [ ]:
X = df[FEATURES]

# XGBoost exige clases 0..n-1: no acepta booleanos ni texto.
# Ver la seccion 9 -- esto tiene consecuencias en el contrato.
y = df[TARGET].astype(bool).astype(int)

X_ent, X_resto, y_ent, y_resto = train_test_split(
    X, y, test_size=0.3, random_state=SEMILLA, stratify=y
)
X_val, X_prueba, y_val, y_prueba = train_test_split(
    X_resto, y_resto, test_size=0.5, random_state=SEMILLA, stratify=y_resto
)

print(f"entrenamiento {len(X_ent):>6}")
print(f"validacion    {len(X_val):>6}")
print(f"prueba        {len(X_prueba):>6}")
print()
print("balance de clases:")
for clase, share in y.value_counts(normalize=True).sort_index().items():
    print(f"  {ETIQUETAS[str(clase)]:<18} {share:.1%}")

## 8. Entrenar

In [ ]:
pipeline = construir_pipeline()
pipeline.fit(X_ent, y_ent)
print("entrenado")

## 9. XGBoost no conserva tus etiquetas — y eso hay que escribirlo en el contrato

Con un estimador de scikit-learn, si entrenas con `True`/`False`, `classes_` te devuelve
`[False, True]` y `predict()` devuelve booleanos. **XGBoost no.** Exige `0..n-1` y rechaza
cualquier otra cosa:

```
ValueError: Invalid classes inferred from unique values of `y`.
            Expected: [0 1], got ['No transportado' 'Transportado']
```

Así que `predict()` devuelve `0` o `1`, y el nombre legible tiene que viajar aparte. Eso es
`class_labels` en el contrato, y es lo que hace que la interfaz muestre *Transportado* en
lugar de *1*.

**El orden de `classes_` importa más de lo que parece.** `predict_proba()` devuelve una fila
de números sin nombres; lo único que dice a qué clase corresponde cada columna es ese orden.
Si el contrato lo declarara al revés, el servicio respondería `200` con la confianza de la
clase equivocada y **nada fallaría**. Por eso el contrato guarda `classes` y el servicio los
compara al arrancar.

In [ ]:
CLASES = pipeline.classes_.tolist()
i_positiva = CLASES.index(CLASE_POSITIVA)

print(f"classes_        : {CLASES}")
print(f"clase positiva  : {CLASE_POSITIVA} ({ETIQUETAS[str(CLASE_POSITIVA)]}), columna {i_positiva}")
print()

ejemplo_proba = pipeline.predict_proba(X_prueba.iloc[[0]])[0]
for clase, p in zip(CLASES, ejemplo_proba):
    print(f"  P({ETIQUETAS[str(clase)]:<18}) = {p:.4f}")

## 10. Medir — y por qué recall

Un falso negativo y un falso positivo no cuestan lo mismo. Un falso negativo es un pasajero
transportado que el sistema reporta **a salvo**: no se despliega ninguna búsqueda para él. Un
falso positivo solo moviliza recursos de más.

Por eso la **métrica de decisión es recall**, con **F1 como filtro de calidad** para que
maximizar recall no degenere en predecir todo positivo. `accuracy`, `precision` y
`specificity` van de contexto.

> **`accuracy` no es la métrica aquí, y conviene decir por qué.** El modelo final tiene *menos*
> accuracy que el anterior (78.3% contra 80.6%) y aun así es mejor para este problema: cambia
> 55 falsos negativos por 95 falsos positivos. Si la Model Card solo mostrara accuracy, ese
> modelo parecería un retroceso. Por eso `metadata.json` lleva `primary_metric` y
> `primary_metric_why`: la tabla de métricas sola no dice cuál mirar.

In [ ]:
def metricas(X_, y_):
    """Las seis metricas, sobre el conjunto que le pases."""
    pred = pipeline.predict(X_)
    proba = pipeline.predict_proba(X_)[:, i_positiva]
    tn, fp, fn, tp = confusion_matrix(y_, pred, labels=CLASES).ravel()
    return {
        "accuracy": round(float(accuracy_score(y_, pred)), 4),
        "precision": round(float(precision_score(y_, pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_, pred)), 4),
        "f1": round(float(f1_score(y_, pred)), 4),
        # Especificidad: de los que NO fueron transportados, cuantos acerto.
        # No viene en sklearn y es la contraparte del recall.
        "specificity": round(float(tn / (tn + fp)) if (tn + fp) else 0.0, 4),
        "roc_auc": round(float(roc_auc_score(y_, proba)), 4),
    }


m_val = metricas(X_val, y_val)
m_prueba = metricas(X_prueba, y_prueba)

print(pd.DataFrame({"validacion": m_val, "prueba": m_prueba}).to_string())

## 11. Comparativa de modelos

Tres candidatos, **el mismo preprocesamiento y el mismo split**. Si cada uno preparara los
datos a su manera, la comparación no diría nada sobre los modelos.

El baseline no es decoración: predecir siempre la clase mayoritaria da ~50% de accuracy en
este dataset y **100% de recall**. Cualquier modelo que no supere eso en F1 no está
aprendiendo nada, y sin la fila del baseline delante es fácil no notarlo.

Se mide sobre **validación**, no sobre prueba. La prueba se toca una vez, al final.

In [ ]:
def evaluar(nombre, estimador, hiperparametros):
    """Entrena un candidato con el pipeline completo y lo mide en validacion."""
    pipe = armar(estimador).fit(X_ent, y_ent)
    pred = pipe.predict(X_val)
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=CLASES).ravel()
    return {
        "modelo": nombre,
        "hiperparametros": hiperparametros,
        "accuracy": round(float(accuracy_score(y_val, pred)), 4),
        "precision": round(float(precision_score(y_val, pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_val, pred)), 4),
        "f1": round(float(f1_score(y_val, pred)), 4),
        "specificity": round(float(tn / (tn + fp)) if (tn + fp) else 0.0, 4),
        "FN": int(fn),
        "FP": int(fp),
    }


sin_ruido = {k: v for k, v in PARAMS_XGB.items() if k not in ("random_state", "eval_metric")}

comparacion = [
    evaluar(
        "Baseline (clase mayoritaria)",
        DummyClassifier(strategy="most_frequent"),
        "strategy='most_frequent'",
    ),
    evaluar(
        "RandomForest",
        RandomForestClassifier(
            n_estimators=500, max_depth=12, min_samples_split=20,
            random_state=SEMILLA, n_jobs=-1,
        ),
        "n_estimators=500, max_depth=12, min_samples_split=20",
    ),
    evaluar(
        "XGBoost",
        XGBClassifier(**PARAMS_XGB),
        ", ".join(f"{k}={v}" for k, v in sin_ruido.items()),
    ),
]

pd.DataFrame(comparacion).set_index("modelo").drop(columns="hiperparametros")

## 12. Experimentos de hiperparámetros

Un parámetro a la vez, todo lo demás fijo en el modelo final. Es más lento de leer que un
`GridSearchCV`, pero contesta una pregunta que la grilla no contesta: **qué hace cada
parámetro por su cuenta**.

Los rangos son los de tu grilla del avance 3, que salieron de las curvas de validación.
Fíjate en las filas de `scale_pos_weight`: es donde se ve el trade-off completo — el recall
sube y la specificity baja, en la misma tabla.

Esta tabla es la que la Model Card muestra en *Experimentos de hiperparámetros*, así que lo
que salga aquí es lo que va a ver quien evalúe el reto.

In [ ]:
BARRIDO = {
    "n_estimators": [75, 100, 125, 150],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05, 0.065, 0.08, 0.1],
    "min_child_weight": [1, 3, 7],
    "subsample": [0.6, 0.8, 1.0],
    # El parametro que decidio el modelo final. Mirar como se mueven recall y
    # specificity en filas contiguas de esta tabla es ver el trade-off entero.
    "scale_pos_weight": [1, 1.2, 1.5, 2],
}

experimentos = []
for parametro, valores in BARRIDO.items():
    for valor in valores:
        experimentos.append(
            evaluar(
                f"XGBoost {parametro}={valor}",
                XGBClassifier(**{**PARAMS_XGB, parametro: valor}),
                f"{parametro}={valor}, resto igual al base",
            )
        )

tabla = pd.DataFrame(experimentos).set_index("modelo").drop(columns="hiperparametros")
print(f"{len(experimentos)} experimentos\n")
tabla

## 13. Comprobación antes de exportar

Tres cosas que, si fallan, hacen que el servicio sirva basura sin dar ningún error.

In [ ]:
# 1. predict() devuelve una clase del target, no un indice inventado.
una_clase = pipeline.predict(X_prueba.iloc[[0]])[0]
assert una_clase in CLASES, f"predict() devolvio {una_clase!r}, que no esta en {CLASES}"

# 2. predict_proba() devuelve una probabilidad por clase, y suman 1.
fila_proba = pipeline.predict_proba(X_prueba.iloc[[0]])[0]
assert len(fila_proba) == len(CLASES), "hay mas probabilidades que clases"
assert abs(fila_proba.sum() - 1.0) < 1e-6, "las probabilidades no suman 1"

# 3. El pipeline acepta UNA fila cruda, que es como lo va a llamar el servicio.
#    Con faltantes incluidos: el servidor no imputa, imputa el artefacto.
con_huecos = X_prueba.iloc[[0]].copy()
con_huecos.loc[:, "Age"] = np.nan
assert pipeline.predict(con_huecos)[0] in CLASES, "el pipeline no tolera un faltante"

print(f"clase    : {una_clase} ({ETIQUETAS[str(una_clase)]})")
print(f"proba    : {dict(zip(CLASES, fila_proba.round(4)))}")
print("OK: el pipeline se comporta como el servicio espera")

---

# Aquí empieza lo que de verdad importa

---

## 14. Exportar el pipeline

Un solo archivo, con todo dentro.

In [ ]:
ruta_pipeline = ARTEFACTOS / "pipeline.joblib"
joblib.dump(pipeline, ruta_pipeline)
hash_artefacto = hashlib.sha256(ruta_pipeline.read_bytes()).hexdigest()[:12]

print(f"{ruta_pipeline.name}  {ruta_pipeline.stat().st_size / 1_000_000:.2f} MB")
print(f"hash: {hash_artefacto}")

# El error clasico: un pipeline que congela /content/... o /Users/tu-nombre/...
# dentro del pickle y falla en el servidor. tests/test_paridad_modelo.py lo
# comprueba tambien, pero cuanto antes se detecte, mejor.
crudo = ruta_pipeline.read_bytes()
sucias = [b for b in (b"/Users/", b"/home/", b"/content/") if b in crudo]
assert not sucias, f"el artefacto arrastra rutas absolutas: {sucias}"
print("sin rutas absolutas")

## 15. Las importancias, traducidas a las features originales

El modelo ve las columnas **después** de derivar y codificar: `TotalSpent_log`,
`HomePlanet_Europa`, `IsChild`... Para el contrato queremos la importancia de las features
que la persona llena.

Dos traducciones, y la segunda es la interesante:

- `HomePlanet_Europa`, `HomePlanet_Mars`… se suman en `HomePlanet`.
- `TotalSpent_log` y `HasSpent` **no vienen de una sola feature**: resumen las cinco cuentas
  de consumo. Su importancia se reparte entre las cinco, usando la tabla `ORIGEN` que declara
  `derivadas.py`.

Esto se hace **aquí y no en el servicio**: el servicio no debería tener que hurgar dentro de
un pipeline anidado para saber qué feature pesa más.

> **Y una segunda tabla, sin repartir.** La atribución de arriba puede esconder justo lo
> interesante: si `HasSpent` —una sola columna— carga con medio modelo, repartirla entre las
> cinco cuentas de consumo hace que la tabla diga *"RoomService, 15%"* cuando lo que el modelo
> usa es *"gastó algo o no"*. Las dos vistas van al contrato: `feature_importances` habla el
> vocabulario del formulario (y es la que alimenta la explicación), y `derived_importances`
> dice qué columnas usa el modelo de verdad.

In [ ]:
def importancias_por_feature(pipe):
    """Devuelve las importancias al vocabulario del contrato."""
    estimador = pipe.named_steps["estimador"]
    nombres = pipe.named_steps["preproceso"].get_feature_names_out()
    pesos = estimador.feature_importances_

    acumulado = {f: 0.0 for f in FEATURES}
    for nombre, peso in zip(nombres, pesos):
        # get_feature_names_out prefija con el nombre del transformer: 'cont__'.
        columna = nombre.split("__", 1)[1]

        if columna in derivadas.ORIGEN:
            duenos = derivadas.ORIGEN[columna]
        else:
            # Una columna one-hot es '<feature>_<categoria>'. Se busca de mas
            # larga a mas corta para que un nombre que es prefijo de otro no se
            # lleve las columnas del otro.
            duenos = [
                f
                for f in sorted(FEATURES, key=len, reverse=True)
                if columna == f or columna.startswith(f + "_")
            ][:1]

        for dueno in duenos:
            acumulado[dueno] += float(peso) / len(duenos)

    total = sum(acumulado.values()) or 1.0
    normalizadas = {k: round(v / total, 4) for k, v in acumulado.items()}
    return dict(sorted(normalizadas.items(), key=lambda kv: kv[1], reverse=True))


def importancias_como_las_ve_el_modelo(pipe):
    """Las importancias SIN repartir: por columna del pipeline.

    Hace falta porque la atribucion de arriba puede esconder lo mas
    interesante. 'HasSpent' es una sola columna derivada; repartirla entre las
    cinco cuentas de consumo hace que la tabla diga "RoomService 15%" cuando lo
    que el modelo usa es "gasto algo o no".

    Las columnas one-hot si se suman --'HomePlanet_Earth' y 'HomePlanet_Europa'
    son la misma variable-- porque ahi la suma no pierde nada.
    """
    estimador = pipe.named_steps["estimador"]
    nombres = pipe.named_steps["preproceso"].get_feature_names_out()

    acumulado = {}
    for nombre, peso in zip(nombres, estimador.feature_importances_):
        columna = nombre.split("__", 1)[1]
        base = next(
            (f for f in sorted(derivadas.NOMINALES, key=len, reverse=True)
             if columna == f or columna.startswith(f + "_")),
            columna,
        )
        acumulado[base] = acumulado.get(base, 0.0) + float(peso)

    total = sum(acumulado.values()) or 1.0
    return dict(
        sorted(
            ((k, round(v / total, 4)) for k, v in acumulado.items()),
            key=lambda kv: kv[1],
            reverse=True,
        )
    )


importancias = importancias_por_feature(pipeline)
importancias_modelo = importancias_como_las_ve_el_modelo(pipeline)

print("Atribuidas a las features del formulario")
print("(es el vocabulario del usuario: lo que alimenta la explicacion)")
for nombre, peso in importancias.items():
    print(f"  {nombre:<16} {peso:>7.2%}  {'#' * int(peso * 55)}")

print()
print("Como las ve el modelo, por columna del pipeline")
print("(aqui se ve si una columna derivada esta cargando con todo)")
for nombre, peso in importancias_modelo.items():
    if peso < 0.005:
        continue
    print(f"  {nombre:<20} {peso:>7.2%}  {'#' * int(peso * 55)}")

## 16. `metadata.json` — el contrato

Un archivo, tres consumidores:

```
                    ┌──▶ el servicio VALIDA la entrada contra él
   metadata.json ───┼──▶ la Model Card se RENDERIZA de él
                    └──▶ la rúbrica de tu reto se EVIDENCIA con él
```

Campos que **no** existían en el modelo de casas y que hacen posible la clasificación:

| Campo | Para qué |
|---|---|
| `task` | El servicio decide si devolver un número o una clase |
| `classes` | El orden de `predict_proba()`; el servicio lo verifica al arrancar |
| `class_labels` | Que la interfaz diga *Transportado* y no *1* |
| `positive_class` | Cuál clase es "la que importa" para recall y para el tablero |
| `confusion_matrix` | Qué tipo de error comete, no solo cuánto se equivoca |
| `dashboard` | Cómo presentarlo: eje de comparación y formato de los números |

Y una feature de tipo `bool`, que el modelo de casas no tenía: el formulario la dibuja como
Sí/No y el servicio acepta `true`, `"true"` y `1` como lo mismo.

In [ ]:
features_contrato = []

for f in NUMERICAS:
    features_contrato.append(
        {
            "name": f,
            "type": "num",
            "min": float(df[f].min()),
            "max": float(df[f].max()),
            "median": float(df[f].median()),
        }
    )

for f in CATEGORICAS:
    vistos = sorted(df[f].dropna().unique().tolist())
    if f == "Cabin_Deck":
        # La cubierta rara se funde antes de entrar al modelo, asi que el
        # contrato no debe ofrecerla: seria un valor que el usuario puede
        # elegir y que el pipeline convierte en otro a sus espaldas.
        vistos = sorted(
            {derivadas.DECK_DESTINO if v == derivadas.DECK_RARA else v for v in vistos}
        )
    features_contrato.append({"name": f, "type": "cat", "allowed": vistos})

for f in BOOLEANAS:
    features_contrato.append({"name": f, "type": "bool"})

matriz = confusion_matrix(y_prueba, pipeline.predict(X_prueba), labels=CLASES)

metadata = {
    "model_version": MODEL_VERSION,
    "task": "clasificacion",
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "artifact_hash": hash_artefacto,
    "algorithm": f"XGBClassifier({', '.join(f'{k}={v}' for k, v in sin_ruido.items())})",
    "target": TARGET,
    "target_transform": None,
    "classes": CLASES,
    "class_labels": ETIQUETAS,
    "positive_class": CLASE_POSITIVA,
    "class_balance": {
        str(k): round(float(v), 4) for k, v in y.value_counts(normalize=True).items()
    },
    "primary_metric": "recall",
    "primary_metric_why": (
        "un falso negativo es un pasajero transportado que el sistema reporta "
        "a salvo, asi que no se despliega ninguna busqueda; un falso positivo "
        "solo moviliza recursos de mas. F1 va de filtro para que maximizar "
        "recall no degenere en predecir todo positivo"
    ),
    "features": features_contrato,
    "splits": {
        "train": int(len(X_ent)),
        "validation": int(len(X_val)),
        "test": int(len(X_prueba)),
    },
    "metrics": {"validation": m_val, "test": m_prueba},
    "confusion_matrix": {"labels": CLASES, "matrix": matriz.tolist()},
    "feature_importances": importancias,
    # Lo mismo, pero sin repartir: es lo que el modelo usa de verdad.
    "derived_importances": importancias_modelo,
    # Como presentarlo. El frontend lo lee para saber contra que comparar una
    # prediccion y como escribir los numeros.
    "dashboard": {
        "group_by": "HomePlanet",
        "value_format": "porcentaje",
        "form_title": "Datos del pasajero",
    },
    "model_comparison": comparacion,
    "hyperparameter_experiments": experimentos,
}

(ARTEFACTOS / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False) + "\n"
)
print(f"metadata.json  {len(json.dumps(metadata)) / 1000:.1f} KB")

## 17. `example.json` — el smoke test

Un input válido, la clase que este modelo le da **ahora mismo**, y sus probabilidades. Es lo
que permite comprobar que el servicio devuelve lo mismo que este notebook.

Ese es `tests/test_paridad_modelo.py`, y es el único test que de verdad importa en este
módulo.

**La fila no puede tener faltantes.** El pipeline sí sabe imputar, pero el contrato es el que
manda en la frontera: el servicio rechaza un `null` con `400`, y el test fallaría por una
razón que no tiene nada que ver con el modelo.

In [ ]:
completas = X_prueba.dropna()
if len(completas) == 0:
    raise RuntimeError(
        "no hay ninguna fila de prueba sin faltantes; elige otra semilla"
    )

caso = completas.iloc[[0]]
proba = pipeline.predict_proba(caso)[0]

ejemplo = {
    "input": {
        k: (v.item() if hasattr(v, "item") else v)
        for k, v in caso.iloc[0].to_dict().items()
    },
    "prediction": int(pipeline.predict(caso)[0]),
    "probabilities": [
        {"class": c, "probability": round(float(p), 4)} for c, p in zip(CLASES, proba)
    ],
    "model_version": MODEL_VERSION,
}

(ARTEFACTOS / "example.json").write_text(
    json.dumps(ejemplo, indent=2, ensure_ascii=False) + "\n"
)
print(json.dumps(ejemplo, indent=2, ensure_ascii=False))

## 18. Resumen

In [ ]:
print("Artefacto exportado en artifacts/")
print(f"  splits          : {metadata['splits']}")
print(f"  recall  val/pru : {m_val['recall']:.4f} / {m_prueba['recall']:.4f}")
print(f"  f1      val/pru : {m_val['f1']:.4f} / {m_prueba['f1']:.4f}")
print(f"  roc_auc prueba  : {m_prueba['roc_auc']:.4f}")
print(f"  sklearn         : {metadata['sklearn_version']}")
print(f"  xgboost         : {metadata['xgboost_version']}")
print(f"  hash            : {hash_artefacto}")
print()
print("  matriz de confusion (prueba):")
print(f"    {'real \\ predicho':<20}", "  ".join(f"{ETIQUETAS[str(c)]:>16}" for c in CLASES))
for i, fila in enumerate(matriz):
    print(f"    {ETIQUETAS[str(CLASES[i])]:<20}", "  ".join(f"{v:>16,}" for v in fila))
print()
print("  importancias:")
for k, v in list(importancias.items())[:5]:
    print(f"    {k:<16} {v}")

## 19. El submission de Kaggle

Aparte del artefacto: la competencia quiere `PassengerId,Transported` con booleanos.

Fíjate en que aquí **se vuelve a usar el mismo pipeline**. No hay una segunda ruta de
preprocesamiento para el test: eso es lo que provoca las diferencias inexplicables entre lo
que mides y lo que Kaggle te puntúa.

In [ ]:
ruta_test = RAIZ / "data" / "test.csv"

if not ruta_test.exists():
    print(f"No hay {ruta_test}; se omite el submission.")
    print("Si lo quieres, sube tambien el test.csv de Kaggle.")
else:
    df_test = preparar(pd.read_csv(ruta_test))
    pred_test = pipeline.predict(df_test[FEATURES])

    submission = pd.DataFrame(
        {
            "PassengerId": df_test["PassengerId"],
            # Kaggle quiere True/False, no 0/1.
            "Transported": pd.Series(pred_test).map({0: False, 1: True}),
        }
    )
    destino = RAIZ / "submission.csv"
    submission.to_csv(destino, index=False)

    print(f"{destino}  ({submission.shape[0]} filas)")
    assert submission.shape == (len(df_test), 2), "el shape no es el que pide Kaggle"
    assert submission["Transported"].dtype == bool, "Transported tiene que ser booleano"
    print(submission.head().to_string(index=False))
    print()
    print(f"tasa de positivos: {submission['Transported'].mean():.1%}")

## 20. Descarga el artefacto

Colab borra tus archivos al cerrar la sesión, así que el artefacto tiene que salir de aquí.

Esta celda te descarga un `artifacts.zip` con los **cuatro** archivos. Descomprímelo en la
raíz de tu proyecto y súbelo con el resto de tu código:

```bash
git add artifacts/
git commit -m "artefacto de clasificacion"
git push
```

Y en tu instancia: `./setup/run sync && ./setup/run restart`.

> Si la descarga automática no arranca —pasa según el navegador—, bájalo a mano desde el panel
> de archivos de la izquierda.

In [ ]:
import zipfile

ESPERADOS = ["pipeline.joblib", "derivadas.py", "metadata.json", "example.json"]
faltan = [n for n in ESPERADOS if not (ARTEFACTOS / n).exists()]

if faltan:
    print("No encuentro estos archivos del artefacto:", ", ".join(faltan))
    print(f"Los busque en: {ARTEFACTOS}")
    print()
    print("Vuelve a correr las celdas de exportacion (de la 14 a la 17).")
else:
    destino = pathlib.Path.cwd() / "artifacts.zip"

    # arcname con el prefijo artifacts/ para que al descomprimir se cree la
    # carpeta. Sin eso salen cuatro archivos sueltos y hay que acomodarlos.
    with zipfile.ZipFile(destino, "w", zipfile.ZIP_DEFLATED) as z:
        for nombre in ESPERADOS:
            z.write(ARTEFACTOS / nombre, arcname=f"artifacts/{nombre}")

    print(f"empaquetado: {destino}  ({destino.stat().st_size / 1_000_000:.2f} MB)")
    for n in zipfile.ZipFile(destino).namelist():
        print(f"  {n}")

    if "google.colab" in sys.modules:
        print()
        try:
            from google.colab import files

            files.download(str(destino))
        except Exception as e:
            print(f"La descarga automatica no funciono ({type(e).__name__}: {e}).")
            print("Bajalo a mano: panel de archivos -> artifacts.zip -> Descargar")
    else:
        print("\nNo estas en Colab: el artefacto ya esta en artifacts/.")

## 21. Qué NO fue al artefacto

Vale la pena decirlo en voz alta:

- **Los datos de entrenamiento.** El artefacto lleva el modelo, no el CSV.
- **Credenciales.** Nunca, en ningún artefacto.
- **Rutas absolutas de esta máquina.** La celda 14 lo comprueba, y el test de paridad también.
- **El `submission.csv`.** Es un entregable de la competencia, no parte del producto.

Y algo que sí fue, y que no estaba en el modelo de casas: **`derivadas.py`**. Si algún día
cambias esa función, tienes que volver a exportar el pipeline. El `artifact_hash` es lo que
te deja notarlo: si el hash del servidor no es el que dice tu `metadata.json`, están sirviendo
otra cosa.

---

## Y ahora, comprueba la costura

En tu instancia, después del `sync`:

```bash
./setup/run restart
./setup/run test paridad_modelo
```

Si el notebook y el servicio devuelven la misma clase **y las mismas probabilidades**, el
modelo cruzó la frontera entero.